In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *

In [2]:
imdb = keras.datasets.imdb

(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=10000
)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [3]:
word_to_index = imdb.get_word_index()

word_to_index = {
    k:(v+3)
    for k,v in word_to_index.items()
}

word_to_index["<PAD>"] = 0
word_to_index["<START>"] = 1
word_to_index["<UNK>"] = 2
word_to_index["<UNUSED>"] = 3

index_to_word = dict(
    [(value,key)
     for (key,value)
     in word_to_index.items()]
)

print("첫 번째 리뷰 복원")
print(
    ' '.join(
        [index_to_word[index]
         for index in x_train[0]]
    )
)

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
첫 번째 리뷰 복원
<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for <UNK> and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also <UNK> to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the <UNK> list i think because the stars that play them all grown up are such a big profile for the whole film 

In [4]:
x_train = pad_sequences(
    x_train,
    maxlen=100
)

x_test = pad_sequences(
    x_test,
    maxlen=100
)

In [5]:
vocab_size = 10000

model = Sequential()

model.add(
    Embedding(
        vocab_size,
        64,
        input_length=100
    )
)

model.add(Flatten())

model.add(
    Dense(
        64,
        activation='relu'
    )
)

model.add(
    Dropout(0.5)
)

model.add(
    Dense(
        1,
        activation='sigmoid'
    )
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=5,
    validation_data=(x_test,y_test)
)

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 26ms/step - accuracy: 0.7680 - loss: 0.4606 - val_accuracy: 0.8427 - val_loss: 0.3481
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - accuracy: 0.9375 - loss: 0.1778 - val_accuracy: 0.8348 - val_loss: 0.3995
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - accuracy: 0.9917 - loss: 0.0344 - val_accuracy: 0.8326 - val_loss: 0.5230
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 0.9987 - loss: 0.0075 - val_accuracy: 0.8344 - val_loss: 0.6151
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 24ms/step - accuracy: 0.9996 - loss: 0.0027 - val_accuracy: 0.8366 - val_loss: 0.6654


In [7]:
results = model.evaluate(
    x_test,
    y_test,
    verbose=2
)

print("\n테스트 정확도 =", results[1])

782/782 - 2s - 3ms/step - accuracy: 0.8366 - loss: 0.6654

테스트 정확도 = 0.8366400003433228


In [8]:
import re

review = """
This movie is fantastic.
The acting was excellent and the story was amazing.
I really enjoyed every moment.
"""

review = re.sub(
    "[^0-9a-zA-Z ]",
    "",
    review
).lower()

review_encoding = []

for w in review.split():

    index = word_to_index.get(w, 2)

    if index <= 10000:
        review_encoding.append(index)
    else:
        review_encoding.append(
            word_to_index["<UNK>"]
        )

test_input = pad_sequences(
    [review_encoding],
    maxlen=100
)

value = model.predict(test_input)

print("\n긍정 확률 =", value[0][0])

if value > 0.5:
    print("긍정적인 리뷰입니다.")
else:
    print("부정적인 리뷰입니다.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step

긍정 확률 = 0.8830948
긍정적인 리뷰입니다.
